# Treatment Effect Estimator — SageMaker Algorithm Usage Guide

This notebook demonstrates how to use the **Treatment Effect Estimator** as a published AWS SageMaker Algorithm. It covers:

1. Uploading your RCT dataset to S3
2. Running a training job
3. Inspecting the training report (hypothesis tests on the training data)
4. Deploying a real-time inference endpoint
5. Calling the endpoint and interpreting every field in the response

---

## What problem does this algorithm solve?

In a **Randomised Controlled Trial (RCT)** subjects are randomly assigned to a treatment arm or a control arm and their outcome is observed. Classical analysis answers one question: *did the treatment work on average?* This service answers a harder, more actionable question:

> **Which subjects are most likely to benefit from treatment, and by how much?**

The algorithm trains a **T-Learner** (two separate ML models — one per arm) to estimate the **Conditional Average Treatment Effect (CATE)** for every individual:

$$\text{CATE}(x) = E[Y(1) - Y(0) \mid X = x]$$

Both **binary** and **continuous** outcome variables are supported:

| Outcome type | CATE definition | Example use case |
|---|---|---|
| Binary (0/1) | $P(\text{event}\mid T=1, x) - P(\text{event}\mid T=0, x)$ | 5-year mortality, conversion, churn |
| Continuous | $E[Y\mid T=1, x] - E[Y\mid T=0, x]$ | Revenue lift, session duration, LTV |

At inference time the endpoint also runs classical **hypothesis tests** on the batch, so you get both the per-subject ML predictions *and* the population-level frequentist analysis in one call.

---

### When is this valid? RCT vs observational data

The CATE estimator is a **meta-learner**: it learns two response surfaces and subtracts them. The subtraction cancels all confounding **only when treatment assignment is independent of subject characteristics** — a guarantee that randomisation provides by design.

| Setting | Treatment assignment | Is this tool valid? |
|---|---|---|
| RCT | Random, controlled by protocol | **Yes** |
| Server-side randomised A/B test | Random, controlled by the platform | **Yes** |
| Quasi-experiment (diff-in-diff, RD) | Exploits a discontinuity or natural experiment | **Only with additional causal assumptions — verify carefully** |
| Observational data (claims, logs, CRM) | Self-selected or administratively assigned | **No — estimates will be biased** |

In observational data, treated and untreated subjects differ systematically *before* any treatment is applied. Those pre-existing differences are indistinguishable from a treatment effect by any model that does not explicitly account for the assignment mechanism. The result is CATE estimates that are biased in magnitude, direction, or both — with no signal in the model output to detect the problem.

For observational settings use methods that model the propensity for treatment: propensity score matching, inverse-probability weighting (IPW), doubly-robust / augmented IPW estimators (AIPW, DR-Learner), or instrumental variable approaches.

---

### Applicability beyond medicine

The only requirements on your data are:

| Column | Type | Meaning |
|--------|------|---------| 
| `TRTMT` | binary (0/1) | whether the subject received the treatment / was in the test group — **must be randomly assigned** |
| `outcome` | binary (0/1) **or** continuous numeric | the event or value you are measuring |
| all others | numeric | covariates / features |

This makes it directly applicable to **product A/B tests**, **marketing experiments**, **policy evaluations**, and any domain where a randomised or quasi-randomised assignment exists.

> **This tool requires experimental data with randomised treatment assignment.**
>
> The T-Learner and S-Learner are meta-learners. They estimate the Conditional Average Treatment Effect (CATE) by differencing two model predictions — one trained on the treated arm, one on the control arm. This difference is a valid estimate of a causal effect **only when treatment assignment is independent of subject characteristics**, i.e. when subjects were randomly assigned to arms (RCT) or when an equivalent design (e.g. a server-side randomised A/B test) guarantees that independence.
>
> **Do not apply this tool to observational data.** In observational settings, treated and untreated subjects differ systematically before any treatment is applied — sicker patients receive treatment, high-value customers are shown promotions, engaged users opt into a feature. The models will absorb those pre-existing differences and mistake them for a treatment effect. The resulting CATE estimates will be biased in magnitude, direction, or both, with no reliable signal in the output to detect the problem.
>
> For observational data use methods that explicitly model the treatment assignment mechanism: propensity score matching, inverse-probability weighting (IPW), doubly-robust estimators (AIPW / DR-Learner), or instrumental variable approaches.

---
## Background: the statistical tests returned by this service

Every time you include the `outcome` column in your request the endpoint runs the following analyses, returned under the `hypothesis_testing` key. The test suite is selected automatically based on the outcome type.

---

### Binary outcome tests

#### 1. Chi-squared test of independence

Tests whether the outcome is **independent** of treatment assignment using a 2×2 contingency table:

| | No event | Event |
|---|---|---|
| **Treatment** | a | b |
| **Control** | c | d |

$$\chi^2 = \sum \frac{(O - E)^2}{E}, \quad df = 1$$

A small p-value (typically < 0.05) gives evidence that treatment and outcome are **not** independent.
Preferred when all expected cell counts are ≥ 5.

#### 2. Fisher's exact test

Computes the exact probability of the observed table under the null of independence. Valid for **any sample size**, including small cells. Preferred when any expected cell count is < 5.

#### 3. Absolute Risk Reduction (ARR) and 95 % CI

$$\text{ARR} = p_{\text{control}} - p_{\text{treatment}}, \qquad
\text{CI} = \text{ARR} \pm 1.96 \cdot \sqrt{\frac{p_t(1-p_t)}{n_t} + \frac{p_c(1-p_c)}{n_c}}$$

Positive ARR means treatment reduces the event rate. If the CI excludes 0 the result is significant at 5 %.

#### 4. Relative Risk (RR) and 95 % CI

$$\text{RR} = \frac{p_t}{p_c}, \qquad
\text{CI} = \exp\!\left(\ln(\text{RR}) \pm 1.96 \cdot \sqrt{\frac{1-p_t}{n_t p_t} + \frac{1-p_c}{n_c p_c}}\right)$$

RR < 1 means treatment reduces the event rate relative to control.

#### 5. Odds Ratio (OR) and 95 % CI

$$\text{OR} = \frac{p_t/(1-p_t)}{p_c/(1-p_c)}$$

OR < 1 means treatment reduces the odds of the event. The natural effect-size measure for logistic regression; for rare outcomes OR ≈ RR.

#### 6. Number Needed to Treat (NNT)

$$\text{NNT} = \frac{1}{\text{ARR}}$$

The number of subjects to treat (on average) to prevent one additional event. NNT < 0 means treatment is harmful on average.

---

### Continuous outcome tests

#### 7. Welch's t-test

Tests $H_0: \mu_{\text{treatment}} = \mu_{\text{control}}$ without assuming equal variances. More robust than Student's t-test for unequal group sizes or variances.

#### 8. Mann-Whitney U test

Non-parametric test for whether one group tends to have larger values than the other. Does not assume normality; preferred for skewed or ordinal outcomes.

#### 9. Mean difference and 95 % CI

$$\Delta\mu = \mu_{\text{treatment}} - \mu_{\text{control}}, \qquad
\text{CI} = \Delta\mu \pm t_{\alpha/2,\nu} \cdot \sqrt{\frac{s_t^2}{n_t} + \frac{s_c^2}{n_c}}$$

Degrees of freedom $\nu$ are computed via the Welch–Satterthwaite formula.

#### 10. Cohen's d

$$d = \frac{\mu_{\text{treatment}} - \mu_{\text{control}}}{s_{\text{pooled}}}$$

Standardised effect size. Rule of thumb: |d| < 0.2 is small, 0.2–0.5 is medium, > 0.8 is large.

---

### 11. C-statistic for benefit (C-for-benefit) — binary only

Evaluates how well the model **ranks** subjects by their individual benefit:

$$\text{C-for-benefit} = P(\hat{\text{CATE}}_i > \hat{\text{CATE}}_j \mid \text{observed benefit}_i > \text{observed benefit}_j)$$

Observed benefit is estimated by matching treated/control pairs. A value of 0.5 = random; higher values indicate genuine discriminative ability. Returned in `metrics.c_for_benefit` for binary outcomes only.

---
## 1. Setup

In [ ]:
%pip install --quiet sagemaker boto3 pandas matplotlib

In [ ]:
import json
import io
import boto3
import pandas as pd
import matplotlib.pyplot as plt
import sagemaker
from sagemaker import AlgorithmEstimator
from sagemaker.predictor import Predictor
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

print(f"SageMaker SDK version: {sagemaker.__version__}")

### 1.1 Configuration

Replace the placeholder values with your own AWS resources.

| Variable | What to put here |
|----------|------------------|
| `ALGORITHM_ARN` | ARN of the published algorithm (visible in *AWS Marketplace → My subscriptions* or *SageMaker → Algorithms*) |
| `ROLE_ARN` | IAM execution role — must have `AmazonSageMakerFullAccess` and S3 read/write on `BUCKET` |
| `BUCKET` | S3 bucket for training data and model artefacts |
| `REGION` | AWS region where the algorithm is published |

In [ ]:
# ── Replace these with your real values ───────────────────────────────────────
ALGORITHM_ARN = "arn:aws:sagemaker:us-east-1:123456789012:algorithm/treatment-effect-estimator-v1"
ROLE_ARN      = "arn:aws:iam::123456789012:role/SageMakerExecutionRole"
BUCKET        = "my-treatment-effect-bucket"
REGION        = "us-east-1"
PREFIX        = "treatment-effect"
# ──────────────────────────────────────────────────────────────────────────────

session   = sagemaker.Session(boto_session=boto3.Session(region_name=REGION))
s3_client = boto3.client("s3", region_name=REGION)

TRAINING_S3_URI = f"s3://{BUCKET}/{PREFIX}/data/"
OUTPUT_S3_URI   = f"s3://{BUCKET}/{PREFIX}/output/"

print(f"Training data : {TRAINING_S3_URI}")
print(f"Output        : {OUTPUT_S3_URI}")

---
## 2. Data Preparation

### 2.1 Expected schema

The algorithm expects a **CSV file with a header row**. Your dataset must include `TRTMT` (treatment indicator) and `outcome`. All other columns are treated as covariates.

The table below shows the reference colon-cancer RCT schema. For other domains, substitute your own features.

| Column | Type | Description |
|--------|------|-------------|
| `sex` | int (0/1) | 1 = male |
| `age` | int | age at enrolment |
| `obstruct` | int (0/1) | colon obstruction by tumour |
| `perfor` | int (0/1) | colon perforation |
| `adhere` | int (0/1) | adherence to nearby organs |
| `nodes` | float | lymph nodes with detectable cancer |
| `node4` | int (0/1) | more than 4 positive lymph nodes |
| `differ_2`, `differ_3` | int (0/1) | one-hot tumour differentiation |
| `extent_2`, `extent_3`, `extent_4` | int (0/1) | one-hot local spread |
| `TRTMT` | int (0/1) | **treatment indicator** |
| `outcome` | int (0/1) **or** float | **target** — binary event or continuous value |

> **Product analytics mapping:** `TRTMT` → experiment assignment flag; `outcome` → conversion (binary) or revenue / session duration (continuous).

---

### Feature engineering is the user's responsibility

**The algorithm performs no feature preprocessing.** All columns must be numeric before the CSV is uploaded to S3 or passed to the inference endpoint. Specifically:

- **Categorical variables** (strings, nominal codes) must be **one-hot encoded** or integer-encoded beforehand. The algorithm will raise an error or produce garbage predictions if string-typed columns are present.
- **Feature scaling** (e.g. `StandardScaler`, `MinMaxScaler`) is **not required** — Random Forests are scale-invariant and produce the same splits regardless of the magnitude of continuous features. You may still apply scaling if your downstream workflow requires it, but it will not improve model accuracy.
- **Missing values** must be imputed or removed before upload. The training script drops rows with `NaN`, but the inference endpoint does not — NaN in an inference request will produce `NaN` predictions without an error message.

The reference dataset (`data/train.csv`, `data/test.csv`) already satisfies all of these requirements: all features are numeric and one-hot encoding has been applied to `differ` and `extent` ahead of time.

In [ ]:
train_df = pd.read_csv("../data/train.csv")
test_df  = pd.read_csv("../data/test.csv")

print(f"Train : {train_df.shape[0]} rows, {train_df['TRTMT'].astype(int).sum()} treated")
print(f"Test  : {test_df.shape[0]} rows")
print(f"Outcome dtype: {train_df['outcome'].dtype}  "
      f"(unique values: {sorted(train_df['outcome'].unique())[:5]})")
train_df.head(3)

### 2.2 Randomisation sanity checks

Before training, confirm your dataset is a valid RCT. Check:
1. **Balance** — treatment probability should be close to the target allocation ratio.
2. **Outcome distribution per arm** — gives a first look at the average treatment effect.

In [ ]:
p_treated = train_df["TRTMT"].astype(int).mean()
print(f"Proportion treated: {p_treated:.1%}  "
      f"({'balanced' if 0.4 < p_treated < 0.6 else 'CHECK BALANCE'})")

is_binary_outcome = set(train_df["outcome"].unique()).issubset({0, 1, 0.0, 1.0})
print(f"Outcome type detected: {'binary' if is_binary_outcome else 'continuous'}")

grp = train_df.groupby(train_df["TRTMT"].astype(int))["outcome"]
if is_binary_outcome:
    rates = grp.mean()
    print(f"Event rate — control: {rates[0]:.1%}, treatment: {rates[1]:.1%}")
    print(f"Naive ARR: {rates[0] - rates[1]:+.1%}")
else:
    means = grp.mean()
    print(f"Mean outcome — control: {means[0]:.4f}, treatment: {means[1]:.4f}")
    print(f"Naive mean difference: {means[1] - means[0]:+.4f}")

### 2.3 Upload to S3

In [ ]:
s3_client.upload_file("../data/train.csv", BUCKET, f"{PREFIX}/data/train.csv")
s3_client.upload_file("../data/test.csv",  BUCKET, f"{PREFIX}/data/test.csv")
print(f"Uploaded to s3://{BUCKET}/{PREFIX}/data/")

---
## 3. Training

### 3.1 Hyperparameters

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `outcome_type` | `binary` \| `continuous` | `binary` | Outcome variable type. Determines whether a `RandomForestClassifier` (binary) or `RandomForestRegressor` (continuous) is used as the base estimator. |
| `learner_type` | `t_learner` \| `s_learner` | `t_learner` | `t_learner` trains two independent models (one per arm); `s_learner` trains one model with treatment as a feature. |
| `n_estimators` | int | `100` | Number of trees in each Random Forest. |
| `max_depth` | int \| `None` | `None` | Maximum tree depth. `None` = fully grown. |
| `min_samples_leaf` | int \| float | `1` | Minimum samples per leaf. A float is interpreted as a fraction of the training set. |
| `random_state` | int | `0` | Random seed for reproducibility. |
| `treatment_col` | str | `TRTMT` | Name of the treatment indicator column in your CSV. |
| `data_filename` | str | `train.csv` | Filename of the training CSV inside the S3 channel. |

### 3.2 Choosing `outcome_type`

- **`binary`** — use when the outcome is a 0/1 event (mortality, conversion, churn). The base estimator is a `RandomForestClassifier`; CATE predictions are probability differences in $[{-1}, 1]$.
- **`continuous`** — use when the outcome is a real-valued measurement (revenue, engagement time, score). The base estimator is a `RandomForestRegressor`; CATE predictions are in the same units as the outcome.

### 3.3 Choosing `t_learner` vs `s_learner`

- **T-Learner** — preferred when treatment effect heterogeneity is expected. Trains one model per arm independently, giving each arm full modelling freedom. Needs enough observations in each arm.
- **S-Learner** — preferred for smaller datasets. Trains one model with `TRTMT` as a feature. More regularised but may shrink the treatment effect estimate toward zero.

In [ ]:
# ── Set outcome_type to match your dataset ────────────────────────────────────
OUTCOME_TYPE = "binary"   # change to "continuous" for real-valued outcomes
# ─────────────────────────────────────────────────────────────────────────────

estimator = AlgorithmEstimator(
    algorithm_arn=ALGORITHM_ARN,
    role=ROLE_ARN,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=OUTPUT_S3_URI,
    sagemaker_session=session,
    hyperparameters={
        "outcome_type":     OUTCOME_TYPE,
        "learner_type":     "t_learner",
        "n_estimators":     "100",
        "max_depth":        "None",
        "min_samples_leaf": "1",
        "random_state":     "0",
        "treatment_col":    "TRTMT",
        "data_filename":    "train.csv",
    },
)

In [ ]:
estimator.fit(
    inputs={"training": TRAINING_S3_URI},
    wait=True,
    logs="All",
)

---
## 4. Training Report

The training job writes `training_report.json` to `/opt/ml/output/data/` (uploaded to S3 automatically). It contains the resolved hyperparameters, dataset shape, and **hypothesis tests on the training data** — a baseline view of the RCT before any model predictions.

In [ ]:
import tarfile, os

job_name      = estimator.latest_training_job.name
output_s3_key = f"{PREFIX}/output/{job_name}/output/output.tar.gz"
local_tar     = "/tmp/sm_output.tar.gz"

s3_client.download_file(BUCKET, output_s3_key, local_tar)
with tarfile.open(local_tar, "r:gz") as tar:
    tar.extractall("/tmp/sm_output/")

with open("/tmp/sm_output/training_report.json") as f:
    training_report = json.load(f)

print(json.dumps(training_report, indent=2))

### 4.1 Interpreting the training report

The cell below pretty-prints the key statistics and works for **both** binary and continuous outcomes.

In [ ]:
ht   = training_report["hypothesis_testing"]
sz   = ht["sample_sizes"]
fx   = ht["effect_sizes"]
otype = ht["outcome_type"]

print(f"=== Training-set RCT summary  (outcome_type={otype}) ===")
print(f"  Treatment arm : n = {sz['treatment_n']}")
print(f"  Control arm   : n = {sz['control_n']}")

if otype == "binary":
    print(f"  Event rate    : treatment={sz['treatment_event_rate']:.1%}, "
          f"control={sz['control_event_rate']:.1%}")
    chi2 = ht["chi_squared_test"]
    print(f"\n  χ² = {chi2['statistic']:.3f}  p = {chi2['p_value']:.4f}")
    print(f"  ARR  = {fx['absolute_risk_reduction']:+.4f}  "
          f"95% CI [{fx['arr_95ci'][0]:+.4f}, {fx['arr_95ci'][1]:+.4f}]")
    print(f"  NNT  = {fx['number_needed_to_treat']}")
else:
    print(f"  Outcome mean  : treatment={sz['treatment_mean']:.4f} ± {sz['treatment_std']:.4f}, "
          f"control={sz['control_mean']:.4f} ± {sz['control_std']:.4f}")
    t_test = ht["welch_t_test"]
    print(f"\n  Welch t = {t_test['statistic']:.3f}  p = {t_test['p_value']:.4f}")
    print(f"  Mean diff = {fx['mean_difference']:+.4f}  "
          f"95% CI [{fx['mean_difference_95ci'][0]:+.4f}, {fx['mean_difference_95ci'][1]:+.4f}]")
    print(f"  Cohen's d = {fx['cohens_d']:.4f}")

print(f"\n  Note: {ht['note']}")

---
## 5. Deploying the Endpoint

> **Cost note:** SageMaker charges per hour the endpoint is running. Delete it when done (§9).

In [ ]:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.t2.medium",
    endpoint_name="treatment-effect-endpoint",
    serializer=CSVSerializer(),
    deserializer=JSONDeserializer(),
)
print(f"Endpoint: {predictor.endpoint_name}")

In [ ]:
# Reconnect after a kernel restart without redeploying:
# predictor = Predictor(
#     endpoint_name="treatment-effect-endpoint",
#     sagemaker_session=session,
#     serializer=CSVSerializer(),
#     deserializer=JSONDeserializer(),
# )

---
## 6. Real-time Inference

The endpoint accepts `text/csv` (with header) or `application/json` (records-oriented array).

Including the `outcome` column is **optional**:
- **Without `outcome`** — returns predictions only (production path).
- **With `outcome`** — also returns `metrics` and `hypothesis_testing` (analyst / evaluation path).

The outcome type (binary vs continuous) is detected automatically from the values in the `outcome` column.

### 6.1 Inference without labels (production)

In [ ]:
X_test = test_df.drop(columns=["outcome"])

buf = io.StringIO()
X_test.to_csv(buf, index=False)

resp_pred = predictor.predict(buf.getvalue(), initial_args={"ContentType": "text/csv"})

print("Keys:", list(resp_pred.keys()))
print(f"ATE  : {resp_pred['avg_treatment_effect']}")
print("First 5 predictions:")
for p in resp_pred["predictions"][:5]:
    print(f"  {p}")

#### Interpreting `treatment_effect` per subject

| Outcome type | CATE > 0 | CATE < 0 | CATE ≈ 0 |
|---|---|---|---|
| **Binary (adverse)** e.g. death | treatment increases risk (harmful) | treatment reduces risk (beneficial) | no individual effect |
| **Binary (positive)** e.g. conversion | treatment increases probability (beneficial) | treatment decreases probability | no individual effect |
| **Continuous** e.g. revenue | treatment increases value | treatment decreases value | no individual effect |

### 6.2 Inference with labels (evaluation / analyst path)

In [ ]:
buf = io.StringIO()
test_df.to_csv(buf, index=False)   # includes 'outcome'

response = predictor.predict(buf.getvalue(), initial_args={"ContentType": "text/csv"})

print("Top-level keys:", list(response.keys()))
print(json.dumps(response, indent=2))

---
## 7. Interpreting the Full Response

### 7.1 `predictions` — per-subject CATE

In [ ]:
cates = [p["treatment_effect"] for p in response["predictions"]]
ate   = response["avg_treatment_effect"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(cates, bins=20, edgecolor="white")
axes[0].axvline(0,   color="red",  linestyle="--", label="no effect")
axes[0].axvline(ate, color="navy", linestyle="-",  label=f"ATE = {ate:.4f}")
axes[0].set_title("Distribution of individual CATEs")
axes[0].set_xlabel("CATE")
axes[0].set_ylabel("count")
axes[0].legend()

n_pos = sum(c > 0 for c in cates)
n_neg = sum(c < 0 for c in cates)
axes[1].bar(["CATE > 0", "CATE < 0"], [n_pos, n_neg], color=["steelblue", "salmon"])
axes[1].set_title("Sign of predicted treatment effect")
axes[1].set_ylabel("count")

plt.tight_layout()
plt.show()
print(f"ATE = {ate}")

### 7.2 `metrics` — ML evaluation

The `metrics` keys differ by outcome type:

| Key | Present when | Meaning |
|---|---|---|
| `c_for_benefit` | binary only | Concordance for benefit; 0.5 = random, higher = better ranking |
| `avg_empirical_risk_reduction` | binary only | Observed ARR = control event rate − treatment event rate |
| `avg_empirical_treatment_effect` | continuous only | Observed mean difference = control mean − treatment mean |

In [ ]:
m = response["metrics"]
print("=== ML Metrics ===")
for k, v in m.items():
    print(f"  {k}: {v}")

if "c_for_benefit" in m:
    cfb = m["c_for_benefit"]
    if cfb > 0.55:
        print(f"\n  C-for-benefit = {cfb:.4f} — meaningful discriminative ability.")
        print("  The model can rank subjects by likelihood of benefit.")
    elif cfb > 0.50:
        print(f"\n  C-for-benefit = {cfb:.4f} — marginal improvement over random.")
    else:
        print(f"\n  C-for-benefit = {cfb:.4f} — not better than random ranking.")

### 7.3 `hypothesis_testing` — classical frequentist analysis

The cell below handles both binary and continuous schemas.

In [ ]:
ht    = response["hypothesis_testing"]
sz    = ht["sample_sizes"]
fx    = ht["effect_sizes"]
otype = ht["outcome_type"]

print(f"=== Hypothesis Tests  (outcome_type={otype}) ===")
print(f"  Treatment : n = {sz['treatment_n']}")
print(f"  Control   : n = {sz['control_n']}")

alpha = 0.05

if otype == "binary":
    print(f"  Event rates: treatment={sz['treatment_event_rate']:.1%}, "
          f"control={sz['control_event_rate']:.1%}")

    chi2 = ht["chi_squared_test"]
    fish = ht["fisher_exact_test"]
    print(f"\n  χ²({chi2['degrees_of_freedom']}) = {chi2['statistic']:.3f}, "
          f"p = {chi2['p_value']:.4f}")
    print(f"  Fisher OR = {fish['odds_ratio']:.4f},  p = {fish['p_value']:.4f}")

    sig = chi2["p_value"] < alpha
    print(f"  → {'Significant' if sig else 'Not significant'} at α={alpha}")

    print(f"\n  ARR = {fx['absolute_risk_reduction']:+.4f}  "
          f"95% CI [{fx['arr_95ci'][0]:+.4f}, {fx['arr_95ci'][1]:+.4f}]")
    print(f"  RR  = {fx['relative_risk']:.4f}  "
          f"95% CI [{fx['relative_risk_95ci'][0]:.4f}, {fx['relative_risk_95ci'][1]:.4f}]")
    print(f"  OR  = {fx['odds_ratio']:.4f}  "
          f"95% CI [{fx['odds_ratio_95ci'][0]:.4f}, {fx['odds_ratio_95ci'][1]:.4f}]")
    print(f"  NNT = {fx['number_needed_to_treat']}")

else:  # continuous
    print(f"  Outcome: treatment={sz['treatment_mean']:.4f} ± {sz['treatment_std']:.4f}, "
          f"control={sz['control_mean']:.4f} ± {sz['control_std']:.4f}")

    t_test = ht["welch_t_test"]
    mwu    = ht["mann_whitney_u_test"]
    print(f"\n  Welch t = {t_test['statistic']:.3f},  p = {t_test['p_value']:.4f}")
    print(f"  Mann-Whitney U = {mwu['statistic']:.1f},  p = {mwu['p_value']:.4f}")

    sig = t_test["p_value"] < alpha
    print(f"  → {'Significant' if sig else 'Not significant'} at α={alpha}")

    print(f"\n  Mean difference = {fx['mean_difference']:+.4f}  "
          f"95% CI [{fx['mean_difference_95ci'][0]:+.4f}, {fx['mean_difference_95ci'][1]:+.4f}]")
    print(f"  Cohen's d = {fx['cohens_d']:.4f}")

print(f"\n  {ht['note']}")

#### Quick decision guide for analysts

**Binary outcomes**

| Scenario | Interpretation | Action |
|---|---|---|
| p < 0.05, ARR CI excludes 0, small NNT | Strong, significant average effect | Ship / adopt the treatment |
| p < 0.05 but NNT > 200 | Statistically significant, negligible practical effect | Weigh cost vs benefit |
| p ≥ 0.05, ARR CI contains 0 | No detectable average effect | Check C-for-benefit for subgroups |
| p ≥ 0.05, C-for-benefit > 0.55 | Heterogeneous effect that averages out | Target likely responders using CATE |
| p ≥ 0.05, C-for-benefit ≈ 0.5 | No signal | Increase sample size or revisit experiment |

**Continuous outcomes**

| Scenario | Interpretation | Action |
|---|---|---|
| p < 0.05, CI excludes 0, \|d\| > 0.2 | Significant and practically meaningful | Ship / adopt the treatment |
| p < 0.05, \|d\| < 0.2 | Statistically significant, small practical effect | Evaluate business impact threshold |
| p ≥ 0.05 | No detectable average effect | Investigate subgroup heterogeneity via CATE |
| p ≥ 0.05, wide CATE distribution | Effect exists but heterogeneous | Target subjects with strongly positive predicted CATE |

---
## 8. Targeting with CATE Predictions

Use the per-subject CATE to rank and selectively treat subjects most likely to benefit.

In [ ]:
results_df = test_df.copy()
results_df["cate"] = cates
results_df = results_df.sort_values("cate")   # ascending: most negative = most benefit

print("Top 10 most likely to benefit (most negative CATE):")
print(results_df[["cate", "TRTMT", "outcome"]].head(10).to_string(index=False))

In [ ]:
results_df["decile"] = pd.qcut(results_df["cate"], 10, labels=False, duplicates="drop")

grp = results_df.groupby(["decile", results_df["TRTMT"].astype(int)])["outcome"].mean().unstack()
grp.columns = ["control", "treatment"]
grp["empirical_effect"] = grp["control"] - grp["treatment"]   # positive = treatment reduces outcome

# Empirical effect label depends on outcome type
effect_label = "Empirical ARR" if is_binary_outcome else "Empirical mean difference (control − treatment)"

plt.figure(figsize=(9, 4))
plt.bar(grp.index, grp["empirical_effect"], color="steelblue")
plt.axhline(0, color="black", linewidth=0.8)
avg_emp = m.get("avg_empirical_risk_reduction") or m.get("avg_empirical_treatment_effect", 0)
plt.axhline(avg_emp, color="red", linestyle="--", label=f"overall = {avg_emp:.4f}")
plt.xlabel("CATE decile  (0 = most predicted benefit)")
plt.ylabel(effect_label)
plt.title("Empirical effect by CATE decile — calibration check")
plt.legend()
plt.tight_layout()
plt.show()
print(grp.round(4).to_string())

A well-calibrated model shows a **monotonically decreasing empirical effect** from left to right: subjects predicted to benefit most (low deciles) should show the largest observed benefit. This is the visual equivalent of the C-for-benefit score.

---
## 9. Cleanup

In [ ]:
predictor.delete_endpoint()
print(f"Deleted endpoint: {predictor.endpoint_name}")

---
## Reference: full response schema

### Binary outcome

```json
{
  "predictions": [{"treatment_effect": -0.042}, "..."],
  "avg_treatment_effect": -0.042,
  "metrics": {
    "c_for_benefit": 0.62,
    "avg_empirical_risk_reduction": 0.057
  },
  "hypothesis_testing": {
    "outcome_type": "binary",
    "sample_sizes": {
      "treatment_n": 245, "control_n": 238, "total_n": 483,
      "treatment_event_rate": 0.3714, "control_event_rate": 0.4286
    },
    "chi_squared_test": {"statistic": 4.62, "p_value": 0.0316, "degrees_of_freedom": 1},
    "fisher_exact_test": {"odds_ratio": 0.7941, "p_value": 0.0334},
    "effect_sizes": {
      "absolute_risk_reduction": 0.0572, "arr_95ci": [0.0048, 0.1097],
      "relative_risk": 0.8667, "relative_risk_95ci": [0.757, 0.992],
      "odds_ratio": 0.7941, "odds_ratio_95ci": [0.639, 0.987],
      "number_needed_to_treat": 17.5
    },
    "note": "..."
  }
}
```

### Continuous outcome

```json
{
  "predictions": [{"treatment_effect": -1.82}, "..."],
  "avg_treatment_effect": -1.82,
  "metrics": {
    "avg_empirical_treatment_effect": -1.95
  },
  "hypothesis_testing": {
    "outcome_type": "continuous",
    "sample_sizes": {
      "treatment_n": 245, "control_n": 238, "total_n": 483,
      "treatment_mean": 12.31, "treatment_std": 4.52,
      "control_mean": 14.26,  "control_std": 4.81
    },
    "welch_t_test": {"statistic": -2.15, "p_value": 0.032},
    "mann_whitney_u_test": {"statistic": 24562.0, "p_value": 0.034},
    "effect_sizes": {
      "mean_difference": -1.95,
      "mean_difference_95ci": [-3.73, -0.17],
      "cohens_d": -0.42
    },
    "note": "..."
  }
}
```

> `metrics` and `hypothesis_testing` are only present when the `outcome` column is included in the request payload.